In [46]:
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings('ignore')

In [47]:
df = pd.read_csv('pzz_fleet_data_full.csv')

with open('pzz_fleet_physical_limits.json', 'r') as f:
    physical_limits = json.load(f)

In [48]:
display(df.head())

display(physical_limits)


,bus_id,temperature,vibration,speed,fuel_consumption,load,outlier_type,is_failure,timestamp
0,30,65.56,18.26,41.10,38.06,50.36,normal,0,2024-01-21 01:42:00
1,2,74.02,25.24,32.78,33.07,62.53,normal,0,2024-05-30 18:04:00
2,29,81.81,20.31,30.07,32.91,28.47,normal,0,2024-11-16 03:47:00
3,74,75.90,14.90,32.42,40.43,39.40,normal,0,2024-10-08 08:08:00
4,98,65.98,11.97,19.71,30.07,25.19,normal,0,2024-03-26 10:03:00


{'temperature': [0, 200],
 'vibration': [0, 100],
 'speed': [0, 150],
 'fuel_consumption': [0, 100],
 'load': [0, 100]}

# A

### Detekcja Outlierów
Porównujemy skuteczność trzech metod detekcji punktów odstających dla każdej kolumny:
1. **IQR (Interquartile Range):** Metoda pudełkowa, wyznacza granice jako `Q1/Q3 +/- 1.5 * IQR`.
2. **Z-score:** Metoda oparta na odchyleniu standardowym, flaguje wartości `|z| > 3`.
3. **Isolation Forest:** Algorytm ML, który izoluje anomalie poprzez losowe podziały danych.

Cel: Znalezienie indeksów wszystkich podejrzanych obserwacji.

In [49]:
features = ['temperature', 'vibration', 'speed', 'fuel_consumption', 'load']

# Słownik do trzymania wyników na później (klucz = nazwa kolumny, wartość = indeksy outlierów)
# Przyjmujemy, że outlierem jest wiersz wykryty przez PRZYNAJMNIEJ JEDNĄ metodę

outliers_map = {} 

print("RAPORT DETEKCJI OUTLIERÓW\n")

for col in features:
    # 1. IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    iqr_mask = (df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))
    iqr_outliers = df[iqr_mask]

    # 2. Z-score
    z_scores = np.abs(stats.zscore(df[col]))
    z_score_outliers = df[z_scores > 3]

    # 3. Isolation Forest
    iso = IsolationForest(contamination=0.03, random_state=2137)
    preds = iso.fit_predict(df[[col]]) 
    iso_outliers = df[preds == -1]

    print(f"[{col.upper()}]")
    print(f"IQR: {len(iqr_outliers)}")
    print(f"Z-score: {len(z_score_outliers)}")
    print(f"Isolation Forest: {len(iso_outliers)}")

    common = set(iqr_outliers.index) & set(z_score_outliers.index) & set(iso_outliers.index)
    print(f"    Wykryte przez WSZYSTKIE 3 metody: {len(common)}")
    
    all_suspects = set(iqr_outliers.index).union(set(z_score_outliers.index)).union(set(iso_outliers.index))
    outliers_map[col] = all_suspects
    
    print("-" * 40)

print("\nGotowe. Podejrzane indeksy zapisane w 'outliers_map'.")

RAPORT DETEKCJI OUTLIERÓW

[TEMPERATURE]
IQR: 339
Z-score: 109
Isolation Forest: 300
    Wykryte przez WSZYSTKIE 3 metody: 109
----------------------------------------
[VIBRATION]
IQR: 350
Z-score: 104
Isolation Forest: 300
    Wykryte przez WSZYSTKIE 3 metody: 104
----------------------------------------
[SPEED]
IQR: 236
Z-score: 94
Isolation Forest: 299
    Wykryte przez WSZYSTKIE 3 metody: 94
----------------------------------------
[FUEL_CONSUMPTION]
IQR: 315
Z-score: 210
Isolation Forest: 298
    Wykryte przez WSZYSTKIE 3 metody: 210
----------------------------------------
[LOAD]
IQR: 215
Z-score: 156
Isolation Forest: 300
    Wykryte przez WSZYSTKIE 3 metody: 156
----------------------------------------

Gotowe. Podejrzane indeksy zapisane w 'outliers_map'.


# B

### Klasyfikacja: Błąd Sensora vs Prawdziwa Anomalia
Weryfikujemy wykryte outliery przy użyciu wiedzy domenowej (`physical_limits.json`).
* **Błąd Sensora:** Wartość wykraczająca poza fizyczne limity urządzenia (np. Temp > 200°C).
* **Prawdziwa Anomalia:** Wartość mieszcząca się w fizyce, ale oznaczona jako outlier statystyczny (np. Temp = 125°C).

Cel: Odróżnienie awarii czujnika (szum) od awarii silnika (sygnał).

In [50]:
for col in features:
    df[f'{col}_status'] = 'Normal'

for col in features:
    limits = physical_limits.get(col)
        
    p_min, p_max = limits[0], limits[1]
    
    # Pobieramy indeksy outlierów, które znaleźliśmy w poprzednim kroku
    suspect_indices = list(outliers_map[col])
    suspect_data = df.loc[suspect_indices, col]
    
    # Błąd Sensora: Poza fizycznymi widełkami
    mask_error = (suspect_data < p_min) | (suspect_data > p_max)
    error_indices = suspect_data[mask_error].index
    
    # Prawdziwa Anomalia: W widełkach, ale statystycznie dziwna
    
    mask_anomaly = ~mask_error
    anomaly_indices = suspect_data[mask_anomaly].index
    
    df.loc[error_indices, f'{col}_status'] = 'SENSOR_ERROR'
    df.loc[anomaly_indices, f'{col}_status'] = 'TRUE_ANOMALY'
    
    print(f"[{col.upper()}] Limity: {p_min} do {p_max}")
    print(f"  - Zidentyfikowane błędy sensora: {len(error_indices)}")
    print(f"  - Zidentyfikowane prawdziwe anomalie: {len(anomaly_indices)}")
    print("-" * 40)


[TEMPERATURE] Limity: 0 do 200
  - Zidentyfikowane błędy sensora: 200
  - Zidentyfikowane prawdziwe anomalie: 139
----------------------------------------
[VIBRATION] Limity: 0 do 100
  - Zidentyfikowane błędy sensora: 200
  - Zidentyfikowane prawdziwe anomalie: 150
----------------------------------------
[SPEED] Limity: 0 do 150
  - Zidentyfikowane błędy sensora: 200
  - Zidentyfikowane prawdziwe anomalie: 99
----------------------------------------
[FUEL_CONSUMPTION] Limity: 0 do 100
  - Zidentyfikowane błędy sensora: 200
  - Zidentyfikowane prawdziwe anomalie: 119
----------------------------------------
[LOAD] Limity: 0 do 100
  - Zidentyfikowane błędy sensora: 200
  - Zidentyfikowane prawdziwe anomalie: 100
----------------------------------------


# C

### Strategie Obsługi i Modelowanie
Trenujemy model `RandomForest` do przewidywania awarii (`is_failure`), porównując 3 strategie czyszczenia:
1. **Drop:** Usunięcie wszystkich wierszy z outlierami (zarówno błędy, jak i anomalie).
2. **Winsorization:** Przycięcie wartości ekstremalnych do 1. i 99. percentyla.
3. **Smart Handling (Recommended):**
    * Błędy fizyczne -> Naprawa (mediana z `X_train`, aby uniknąć Data Leakage).
    * Prawdziwe anomalie -> Zachowanie wartości + dodanie flagi `_is_high`.

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

X = df[features] 
y = df['is_failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2137)

results = {}

TRENOWANIE MODELI DLA RÓŻNYCH STRATEGII


In [52]:
# STRATEGIA 1: Usuwanie outlierów ze zbioru treningowego
# Usuwamy wiersze, które w kroku b oznaczyliśmy jako SENSOR_ERROR lub TRUE_ANOMALY

train_indices = X_train.index
mask_to_drop = pd.Series(False, index=train_indices)

for col in features:
    # Jeśli status NIE JEST 'Normal', to zaznacz do usunięcia
    status_col = f'{col}_status' # mamy te kolumny w df
    # Sprawdzamy status dla wierszy z treningu
    mask_to_drop |= (df.loc[train_indices, status_col] != 'Normal')

X_train_drop = X_train[~mask_to_drop]
y_train_drop = y_train[~mask_to_drop]

model_drop = RandomForestClassifier(random_state=2137)
model_drop.fit(X_train_drop, y_train_drop)

pred_drop = model_drop.predict(X_test)
f1_drop = f1_score(y_test, pred_drop)
results['1. Drop'] = f1_drop
print(f"Strategia DROP -> F1 Score: {f1_drop:.4f} (Liczba usuniętych wierszy w train: {mask_to_drop.sum()})")

Strategia DROP -> F1 Score: 0.0000 (Liczba usuniętych wierszy w train: 438)


Wyczyśliśmy zbiór trenigowy ze wszystkich awari, tzn model uczył się tylko na zdrowych danych, dlatego zgadywał że wszystko jest zdrowe.

In [53]:
# STRATEGIA 2: Winsoryzacja (clip do 1./99. percentyla)

X_train_win = X_train.copy()
X_test_win = X_test.copy()

for col in features:
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)
    # Clipujemy
    X_train_win[col] = X_train_win[col].clip(lower, upper)
    X_test_win[col] = X_test_win[col].clip(lower, upper)

model_win = RandomForestClassifier(random_state=2137)
model_win.fit(X_train_win, y_train)
pred_win = model_win.predict(X_test_win)
f1_win = f1_score(y_test, pred_win)

results['2. Winsorize'] = f1_win
print(f"Strategia WINSORIZE -> F1 Score: {f1_win:.4f}")

Strategia WINSORIZE -> F1 Score: 1.0000


In [54]:
# STRATEGIA 3: 

# mediany na zbiorze treningowym, żeby potem robić podmianki za anomalie
train_medians = {}
for col in features:
    train_medians[col] = X_train[col].median()

print(f"Mediany wyuczone na zbiorze treningowym: {train_medians}")

def clean_smart_v2(data_subset, full_df_status, trained_medians):
    data_out = data_subset.copy()
    
    for col in features:
        current_status = full_df_status.loc[data_out.index, f'{col}_status']
        
        mask_err = (current_status == 'SENSOR_ERROR')
        fill_val = trained_medians[col] 
        data_out.loc[mask_err, col] = fill_val
        
        thresh = X_train[col].quantile(0.95) 
        data_out[f'{col}_is_high'] = (data_out[col] > thresh).astype(int)
        
    return data_out

# Oba zbiory czyścimy używając wiedzy z TRAIN (train_medians)
X_train_smart = clean_smart_v2(X_train, df, train_medians)
X_test_smart  = clean_smart_v2(X_test, df, train_medians)

model_smart = RandomForestClassifier(random_state=42)
model_smart.fit(X_train_smart, y_train)
pred_smart = model_smart.predict(X_test_smart)

f1_smart = f1_score(y_test, pred_smart)

print(f"\nPoprawiona Strategia SMART -> F1 Score: {f1_smart:.4f}")

Mediany wyuczone na zbiorze treningowym: {'temperature': 76.9, 'vibration': 15.1, 'speed': 34.935, 'fuel_consumption': 34.92, 'load': 50.245000000000005}

Poprawiona Strategia SMART -> F1 Score: 1.0000


Klipowanie oraz dodanie nowej kategori zadziałało równie dobrze wzgl F1-score

# D

### Pipeline Produkcyjny
Implementacja funkcji decyzyjnej działającej w trybie wnioskowania (na pojedynczym odczycie).
System nie trenuje modelu, lecz wykorzystuje ustalone progi:
1. Jeśli wartość poza zakresem JSON → **Napraw** (status `FIXED`).
2. Jeśli wartość powyżej progu z IQR → **Alarmuj** (status `ALERT`).

In [55]:
# progi "alarmowe" IQR sesnownie działało
alarm_thresholds = {}
for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    alarm_thresholds[col] = Q3 + 1.5 * IQR

# medainy do łatania dziur
fix_values = {}
for col in features:
    fix_values[col] = df[col].median()

In [56]:
def production_pipeline(reading, limits, thresholds, fixes):
    """
    reading: słownik z jednym odczytem np. {'temp': 500, 'speed': 50}
    limits: słownik z zakresami min/max (z JSON)
    thresholds: słownik z progami alarmowymi (Q3 + 1.5 IQR)
    fixes: słownik z wartościami do naprawy (mediany)
    """
    
    clean_reading = reading.copy()
    alerts = []
    actions = []
    
    for col in features:
        if col not in reading:
            continue
            
        val = reading[col]
        
        p_min, p_max = limits.get(col, [-float('inf'), float('inf')])
        alarm_level = thresholds.get(col, float('inf'))
        fix_val = fixes.get(col, 0)
        
        if val < p_min or val > p_max:
            # NAPRAWIAMY jak poza widełkami
            clean_reading[col] = fix_val 
            actions.append(f"FIXED: {col} value {val} is impossible. Replaced with {fix_val:.1f}")
            continue 
            
        if val > alarm_level:
            # Nie zmieniamy danych (bo są prawdziwe), ale alarmujemy
            alerts.append(f"ALERT: {col} value {val} is CRITICAL (Threshold: {alarm_level:.1f})")

    # --- WYNIK ---
    status = "OK"
    if actions: status = "WARNING (Data Fixed)"
    if alerts:  status = "CRITICAL (Anomaly Detected)"
    
    return {
        "bus_id": reading.get("bus_id", "Unknown"),
        "final_status": status,
        "clean_data": clean_reading,
        "logs": actions + alerts
    }

In [57]:
stream_data = [
    {'bus_id': 101, 'temperature': 88, 'speed': 45},   # 1. Wszystko w normie
    {'bus_id': 102, 'temperature': 500, 'speed': 0},   # 2. Błąd sensora (powinien naprawić na medianę)
    {'bus_id': 103, 'temperature': 125, 'speed': 20},  # 3. Przegrzanie (powinien dać ALERT)
]

print("--- URUCHOMIENIE PIPELINE'U ---")

for raw_packet in stream_data:
    result = production_pipeline(
        raw_packet, 
        physical_limits, 
        alarm_thresholds, 
        fix_values
    )
    
    print(f"\nBus {result['bus_id']} Status: [{result['final_status']}]")
    if result['logs']:
        for log in result['logs']:
            print(f"  -> {log}")
    print(f"Dane: {result['clean_data']}")

--- URUCHOMIENIE PIPELINE'U ---

Bus 101 Status: [OK]
Dane: {'bus_id': 101, 'temperature': 88, 'speed': 45}

Bus 102 Status: [WARNING (Data Fixed)]
  -> FIXED: temperature value 500 is impossible. Replaced with 76.9
Dane: {'bus_id': 102, 'temperature': 76.91, 'speed': 0}

Bus 103 Status: [CRITICAL (Anomaly Detected)]
  -> ALERT: temperature value 125 is CRITICAL (Threshold: 99.6)
Dane: {'bus_id': 103, 'temperature': 125, 'speed': 20}
